# Lending Club — Credit Risk & Loan Default Prediction
## Part 1: Exploratory Data Analysis (EDA)

**Goal:** Understand the structure of the dataset, identify the most relevant variables for predicting loan default, and prepare the data for modeling.

**Dataset:** Lending Club Loan Data 2007–2018 (~2M loans, 151 columns)  
**Target variable:** `loan_status` → binary: `default = 1` (loan was not repaid), `default = 0` (loan was fully paid)


## 0. Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

---
## 1. Column Selection

### Why we don't use all 151 columns

The dataset has 151 columns, but we cannot and should not use all of them. The columns fall into three groups:

**Group 1 — Columns we USE (origination features)**  
These are variables the bank knows *before* or *at the moment* of approving a loan: the applicant's income, their FICO score, how much debt they have, the loan amount, etc. This information exists before we know whether they will repay or not. These are the only variables a real model could use in production.

**Group 2 — Columns we EXCLUDE due to data leakage**  
These are variables that only exist *after* the loan has already happened:  
- `total_pymnt` — how much the borrower paid in total → only known after the fact  
- `recoveries` — how much the bank recovered → only exists if default already occurred  
- `last_pymnt_amnt` — the last payment made → post-origination information  

If the model sees these columns, it would get 99% accuracy but would be completely useless in real life — it learned to cheat using future information. This is called **data leakage** and it is one of the most common and dangerous mistakes in machine learning projects.

**Group 3 — Columns we EXCLUDE due to noise or missing data**  
Variables like `url`, `id`, `member_id`, columns that are 80-100% empty, or columns that are constant across all rows — they add no predictive value.

In [2]:
# These are the 32 origination-only columns we selected.
# Every column here was available at the time the loan was issued.

columns = [
    'issue_d',         # date the loan was issued (used for temporal analysis)
    # Loan characteristics
    'loan_amnt',       # amount requested
    'term',            # 36 or 60 months
    'int_rate',        # interest rate assigned by Lending Club
    'installment',     # monthly payment amount
    'grade',           # risk grade assigned by LC (A to G)
    'sub_grade',       # more granular version of grade
    'purpose',         # reason for the loan
    # Applicant profile
    'emp_length',      # years of employment
    'home_ownership',  # own / mortgage / rent
    'annual_inc',      # annual income
    'verification_status',  # whether income was verified
    'addr_state',      # US state
    'application_type',# individual or joint application
    # Credit history
    'dti',             # debt-to-income ratio
    'fico_range_low',  # FICO score lower bound
    'fico_range_high', # FICO score upper bound
    'delinq_2yrs',     # delinquencies in the last 2 years
    'earliest_cr_line',# date of first credit line (length of credit history)
    'inq_last_6mths',  # credit inquiries in the last 6 months
    'open_acc',        # number of open credit accounts
    'pub_rec',         # number of derogatory public records
    'revol_bal',       # total revolving balance
    'revol_util',      # revolving credit utilization rate
    'total_acc',       # total number of credit accounts ever
    'mort_acc',        # number of mortgage accounts
    'pub_rec_bankruptcies', # number of bankruptcies
    'bc_util',         # bankcard utilization rate
    'pct_tl_nvr_dlq',  # % of accounts never delinquent
    'percent_bc_gt_75',# % of bankcards with utilization > 75%
    'num_accts_ever_120_pd', # accounts ever 120+ days past due
    # Target
    'loan_status'
]

print('Number of selected columns:', len(columns))
print('Columns excluded from the original 151:', 151 - len(columns))


Number of selected columns: 32
Columns excluded from the original 151: 119


**Finding:** We selected 31 out of 151 columns. The 120 excluded columns are either post-origination data (leakage risk), identifiers with no predictive value, or columns with too many missing values to be useful.

---
## 2. Loading the Dataset

### Why we don't load the full file

The CSV file is 1.6 GB. Loading it completely into memory would take a long time.
Instead, we use `usecols` to load only the 32 columns we need — pandas ignores the other 119 entirely.

### Why we use stratified sampling instead of `nrows`

A naive `nrows=150000` takes the **first** 150,000 rows. Because the Lending Club dataset
is sorted chronologically, this means we would only get loans from **2007–2010** — the early years
with very few loans and very different market conditions.

Instead, we use **stratified sampling by year**: we sample ~13,000 loans from each year
between 2007 and 2018, giving us a final dataset of ~143,000 rows that is representative
of the entire 11-year period. This is critical for any temporal analysis we want to do later.


In [ ]:
print('Loading CSV (selected columns only — ignoring other 119)...')
df_full = pd.read_csv(
    'accepted_2007_to_2018Q4.csv',
    usecols=columns,
    low_memory=False
)
print(f'Full load: {df_full.shape[0]:,} rows')

# Remove 'Current' loans — they are still active and have no known outcome yet.
# Including them would mean trying to predict something we don't know the answer to.
df_full = df_full[df_full['loan_status'] != 'Current'].copy()

# Parse issue_d to extract year for stratification
df_full['issue_d'] = pd.to_datetime(df_full['issue_d'], format='%b-%Y', errors='coerce')
df_full['_year']   = df_full['issue_d'].dt.year
df_full = df_full[df_full['_year'].between(2007, 2018)]

print('\nLoans per year in the full dataset:')
print(df_full['_year'].value_counts().sort_index())

# Stratified sample: 13,000 loans per year → ~143,000 total
# For years with fewer than 13,000 loans we take all of them.

# N_PER_YEAR = 10000
N_PER_YEAR = 10500

df = (
    df_full
    .groupby('_year', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), N_PER_YEAR), random_state=42))
    .reset_index(drop=True)
)

# Drop the helper column now that sampling is done
if '_year' in df.columns:
    df = df.drop(columns=['_year'])

print(f'\nSampled dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('\nLoans per year in the sample:')
print(df['issue_d'].dt.year.value_counts().sort_index())


Loading CSV (selected columns only — ignoring other 119)...


**Finding:** After removing "Current" loans and applying stratified sampling by year, we have a dataset with a known outcome for every loan, covering the full 2007–2018 period. Each year contributes up to 13,000 loans, avoiding the bias of taking only the earliest records. This makes the sample representative both in terms of loan outcomes and market conditions across time.

---
## 3. First Look at the Data

### Why we do this

Before any analysis, we need to understand the structure of the data: what types of variables we have, whether the values look reasonable, and whether there are any obvious issues. Skipping this step often leads to errors later in the pipeline.

In [ ]:
df.head()

**Finding:** The dataset has a mix of numerical variables (loan_amnt, int_rate, dti) and categorical variables stored as text (grade, purpose, home_ownership, emp_length). Some columns already show NaN values in the first rows, which we will address in the missing values section.

In [ ]:
df.dtypes

**Finding:** Most numerical variables are already stored as float64 or int64, which is correct. Variables like `grade`, `term`, and `emp_length` are stored as `object` (text) — we will need to convert them to numbers before modeling. This is part of feature engineering.

In [ ]:
df.describe()

**Finding:** Two important observations from the descriptive statistics:
- `annual_inc` has an extremely high maximum value compared to its median — there are outliers of very high incomes in the dataset
- `dti` also has some very high values in the upper range

These outliers could distort the models if not handled. We will visualize them with boxplots in Section 8.

---
## 4. Missing Values

### Why we handle missing values before analyzing

Most machine learning models cannot handle NaN values — they will throw an error or produce incorrect results. Beyond that, how we fill in missing values is a modeling decision that needs justification. Different strategies are appropriate for different types of variables.

In [ ]:
# Count and calculate the percentage of missing values per column
missing = df.isnull().sum()
missing_pct = round(df.isnull().sum() / len(df) * 100, 1)

missing_table = pd.DataFrame({'missing': missing, 'percentage (%)': missing_pct})
missing_table = missing_table[missing_table['missing'] > 0]
missing_table = missing_table.sort_values('percentage (%)', ascending=False)

print(missing_table)

**Finding:** Most columns have less than 10% missing values, which is manageable. The most affected columns are `bc_util`, `pct_tl_nvr_dlq`, and `percent_bc_gt_75`. Count-based columns like `mort_acc` and `pub_rec_bankruptcies` also have missing values, but in this context a missing value most likely means zero — the applicant simply never had that type of event.

In [ ]:
# Strategy 1: fill continuous numerical variables with the median
# We use the median instead of the mean because it is more robust to outliers —
# an extreme income value will not distort the imputation.
df['revol_util']         = df['revol_util'].fillna(df['revol_util'].median())
df['bc_util']            = df['bc_util'].fillna(df['bc_util'].median())
df['pct_tl_nvr_dlq']     = df['pct_tl_nvr_dlq'].fillna(df['pct_tl_nvr_dlq'].median())
df['percent_bc_gt_75']   = df['percent_bc_gt_75'].fillna(df['percent_bc_gt_75'].median())

# Strategy 2: fill count variables with 0
# If the field is empty it means the applicant never had that event, so 0 is the correct value.
df['mort_acc']               = df['mort_acc'].fillna(0)
df['pub_rec_bankruptcies']   = df['pub_rec_bankruptcies'].fillna(0)
df['num_accts_ever_120_pd']  = df['num_accts_ever_120_pd'].fillna(0)

# Strategy 3: fill emp_length with the most frequent value (mode)
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].mode()[0])

print('Remaining missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

**Finding:** We applied three different imputation strategies depending on the nature of each variable. The dataset is now clean and ready for analysis. Using different strategies for different variable types is a more principled approach than simply applying the same method to everything.

---
## 5. Creating the Binary Target

### Why we need to convert loan_status to a binary variable

The original `loan_status` column has 6 different text values. Classification models need a numerical binary target (0 or 1). We need to decide which statuses represent "default" and which represent "no default" — and we need to justify that decision.

In [ ]:
# First, let's see all the possible values and how frequent they are
print(df['loan_status'].value_counts())

**Finding:** There are 6 loan statuses. `Fully Paid` is clearly the "good" outcome. All others (`Charged Off`, `Late`, `In Grace Period`, `Default`) represent some degree of non-payment that costs the bank money.

In [ ]:
# We define default = 1 for any loan that was not fully repaid.
# We include 'In Grace Period' because even though the borrower has not formally defaulted yet,
# it is an early warning signal that a bank would want to detect in advance.

default_statuses = [
    'Charged Off',
    'Late (16-30 days)',
    'Late (31-120 days)',
    'In Grace Period',
    'Default'
]

df['default'] = df['loan_status'].isin(default_statuses).astype(int)

# Create a labeled version for plots
df['default_label'] = df['default'].map({0: 'No Default', 1: 'Default'})

print(df['default'].value_counts())
print()
print('Default rate:', round(df['default'].mean() * 100, 1), '%')

**Finding:** The default rate is approximately 20%, meaning 1 in every 5 loans ended in some form of non-payment. This imbalance between the two classes is an important characteristic of the dataset that we will analyze in the next section.

---
## 6. Class Imbalance

### Why class imbalance matters

When one class is much more frequent than the other, a naive model can achieve high accuracy by simply always predicting the majority class. In this case, a model that always predicts "no default" would be correct 80% of the time — but it would never detect a single default, which is exactly what the bank needs.

This is why **accuracy is not a good metric here**. Instead, we will use AUC-ROC, which measures how well the model separates the two classes regardless of their proportions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: breakdown of all loan statuses
status_counts = df['loan_status'].value_counts().reset_index()
status_counts.columns = ['loan_status', 'count']

sns.barplot(data=status_counts, y='loan_status', x='count', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of loan_status', fontweight='bold')
axes[0].set_xlabel('Number of loans')
axes[0].set_ylabel('')

# Right: binary target
sns.countplot(
    data=df,
    x='default_label',
    palette={'No Default': '#2ecc71', 'Default': '#e74c3c'},
    ax=axes[1]
)
axes[1].set_title('Binary target', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Number of loans')

plt.suptitle('Class Imbalance: ~80% no default vs ~20% default', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Finding:** The imbalance is clear — there are roughly 4 times more "no default" loans than defaults. This confirms we need to:
1. Use `class_weight='balanced'` when training models so they pay equal attention to both classes
2. Evaluate models with **AUC-ROC** and **precision-recall**, not accuracy
3. Consider tuning the decision threshold — the default 0.5 cutoff may not be optimal

---
## 7. Feature Engineering

### Why we need to transform some variables

Several variables are stored as text and need to be converted to numbers before we can use them in models or calculate correlations. We also create new derived variables that are more informative than the originals.

In [ ]:
# term: extract the number of months from the text
# '36 months' → 36
df['term_months'] = df['term'].str.replace(' months', '').str.strip().astype(float)

# emp_length: convert employment years to a number
# '< 1 year' → 0 | '10+ years' → 10 | '3 years' → 3
df['emp_length'] = df['emp_length'].str.replace('< 1 year', '0')
df['emp_length'] = df['emp_length'].str.replace('10+ years', '10')
df['emp_length'] = df['emp_length'].str.replace(' years', '').str.replace(' year', '')
df['emp_length_yrs'] = pd.to_numeric(df['emp_length'], errors='coerce')

# fico_avg: average of the FICO score range
# Using the midpoint gives us a single representative credit score
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# grade_num: convert letter grade to an ordinal number
# A=1 (lowest risk) to G=7 (highest risk)
# This preserves the natural ordering of the grades
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade_num'] = df['grade'].map(grade_map)

# issue_d: already parsed to datetime at load time.
# We extract year and month as separate numeric columns for easy grouping.
df['issue_year']  = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month

print('New columns created:')
df[['term_months', 'emp_length_yrs', 'fico_avg', 'grade_num', 'issue_year', 'issue_month']].head()


**Finding:** We successfully converted 5 text/date variables into usable numbers. The grade conversion (A=1 to G=7) preserves the natural risk ordering — this is important because it allows the model to treat grade as an ordered variable rather than arbitrary categories. `issue_year` and `issue_month` let us group loans by time period for temporal analysis, and confirm that our stratified sample covers the full 2007–2018 range evenly.

---
## 8. Default Rate by Categorical Variables

### Why we analyze default rate by category

If the default rate is very different across the categories of a variable, that variable has predictive power — the model can use it to distinguish risky loans from safe ones. We use `groupby` to calculate the default rate per category.

In [ ]:
# ── Default rate by GRADE ─────────────────────────────────────
# If grade is a good predictor, we expect a clear gradient: A (lowest) → G (highest)

grade_default = df.groupby('grade')['default'].mean().reset_index()
grade_default.columns = ['grade', 'default_rate']
grade_default = grade_default.sort_values('grade')

plt.figure(figsize=(8, 5))
sns.barplot(data=grade_default, x='grade', y='default_rate', palette='RdYlGn_r')
plt.title('Default rate by loan grade', fontweight='bold')
plt.ylabel('Default rate')
plt.xlabel('Grade (A = lowest risk, G = highest risk)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.show()

print(grade_default)

**Finding:** The gradient is very clear and consistent — default rate increases steadily from Grade A (~5%) to Grade G (~30%+). This confirms that `grade` is the single strongest predictor in the dataset. Lending Club assigns these grades using dozens of internal risk variables, so in a sense, grade already summarizes a lot of information in a single feature.

In [ ]:
# ── Default rate by PURPOSE ───────────────────────────────────
# We expect some loan purposes to be riskier than others.
# The red dashed line shows the overall average default rate.

purpose_default = df.groupby('purpose')['default'].mean().reset_index()
purpose_default.columns = ['purpose', 'default_rate']
purpose_default = purpose_default.sort_values('default_rate')

average = df['default'].mean()

plt.figure(figsize=(9, 6))
sns.barplot(data=purpose_default, y='purpose', x='default_rate', color='steelblue')
plt.axvline(average, color='red', linestyle='--', linewidth=1.5,
            label=f'Overall average: {average:.1%}')
plt.title('Default rate by loan purpose', fontweight='bold')
plt.xlabel('Default rate')
plt.ylabel('')
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.legend()
plt.tight_layout()
plt.show()

**Finding:** Loan purpose has a notable impact on default rate. `small_business` is clearly the riskiest — this makes sense because small businesses have a high failure rate. On the other end, `wedding` and `major_purchase` have the lowest default rates, likely because borrowers in these categories tend to have greater financial stability. Everything to the right of the dashed line has above-average risk.

In [ ]:
# ── Default rate by HOME OWNERSHIP ───────────────────────────
# Hypothesis: homeowners (OWN) should be the least risky,
# renters (RENT) the most risky.

home_default = (df[df['home_ownership'].isin(['OWN', 'MORTGAGE', 'RENT'])]
                .groupby('home_ownership')['default']
                .mean()
                .reset_index())
home_default.columns = ['home_ownership', 'default_rate']
home_default = home_default.sort_values('default_rate')

plt.figure(figsize=(7, 4))
sns.barplot(
    data=home_default,
    x='home_ownership',
    y='default_rate',
    palette=['#2ecc71', '#f39c12', '#e74c3c']
)
plt.axhline(average, color='black', linestyle='--', linewidth=1.5,
            label=f'Overall average: {average:.1%}')
plt.title('Default rate by home ownership', fontweight='bold')
plt.ylabel('Default rate')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.legend()
plt.tight_layout()
plt.show()

**Finding:** The hypothesis holds: OWN < MORTGAGE < RENT in terms of default rate. Homeowners have the lowest risk — owning a home is a sign of financial stability and accumulated assets. Renters have the highest default rate, likely because they have fewer assets and less financial stability. While the differences are not huge, they are consistent and add predictive value to the model.

In [ ]:
# ── Default rate by VERIFICATION STATUS ──────────────────────
# Intuition would say verified borrowers should default less.
# Let's check if that is actually true.

verif_default = df.groupby('verification_status')['default'].mean().reset_index()
verif_default.columns = ['verification_status', 'default_rate']
verif_default = verif_default.sort_values('default_rate')

plt.figure(figsize=(7, 4))
sns.barplot(
    data=verif_default,
    x='verification_status',
    y='default_rate',
    palette=['#2ecc71', '#f39c12', '#e74c3c']
)
plt.axhline(average, color='black', linestyle='--', linewidth=1.5,
            label=f'Overall average: {average:.1%}')
plt.title('Default rate by income verification status', fontweight='bold')
plt.ylabel('Default rate')
plt.xlabel('')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.legend()
plt.tight_layout()
plt.show()

**Finding:** This is a counterintuitive result — "Verified" borrowers actually have a *higher* default rate than "Not Verified" ones. This does not mean verification is bad. What is happening is that Lending Club verifies borrowers it considers riskier — it is not a random verification, it is a selective one. This is called a **confounder**: the variable appears to indicate one thing but actually reflects a different underlying dynamic. The variable is still useful for the model, but it cannot be interpreted causally.

---
## 9. Distributions of Numerical Variables

### Why we compare distributions between the two classes

If a variable has very different distributions for defaulters vs non-defaulters, the model can use it to separate the two groups. We use KDE (Kernel Density Estimate) plots to visualize this — the more separated the two curves, the more predictive power that variable has.

In [ ]:
variables = [
    ('fico_avg',   'FICO Score'),
    ('int_rate',   'Interest Rate (%)'),
    ('dti',        'DTI Ratio'),
    ('annual_inc', 'Annual Income (USD)'),
    ('revol_util', 'Revolving Utilization (%)'),
    ('loan_amnt',  'Loan Amount (USD)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (col, title) in zip(axes, variables):
    # Cap at 99th percentile so outliers don't compress the plot
    cap = df[col].quantile(0.99)
    data = df[df[col] <= cap]

    sns.kdeplot(
        data=data,
        x=col,
        hue='default_label',
        palette={'No Default': '#2ecc71', 'Default': '#e74c3c'},
        fill=True,
        alpha=0.3,
        linewidth=2,
        ax=ax
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Distributions by class: Default vs No Default\nMore separated curves = more predictive power',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Finding:** The plots reveal clear differences between the two groups:
- **FICO Score:** defaults have lower scores — lower credit score means higher historical risk
- **Interest Rate:** defaults have higher rates — Lending Club charges more precisely because they are riskier
- **DTI Ratio:** defaults have a higher debt-to-income ratio — more debt relative to income creates more financial stress
- **Annual Income:** defaults tend to have lower incomes, though there is more overlap between the curves
- **Revolving Utilization:** defaults use a higher percentage of their available credit
- **Loan Amount:** the curves overlap a lot — loan amount alone does not separate the two classes well

Variables with the most separated curves (FICO, int_rate, dti) will be the strongest predictors in the models.

In [ ]:
# Boxplots show the median and spread, and make outliers visible

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, col in zip(axes, ['fico_avg', 'int_rate', 'dti']):
    cap = df[col].quantile(0.99)
    data = df[df[col] <= cap]
    sns.boxplot(
        data=data,
        x='default_label',
        y=col,
        palette={'No Default': '#2ecc71', 'Default': '#e74c3c'},
        width=0.5,
        ax=ax
    )
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Boxplots: median and spread by class', fontweight='bold')
plt.tight_layout()
plt.show()

**Finding:** The boxplots confirm what the KDE plots showed. The median lines are clearly at different positions for the two groups in all three variables. Additionally, we can see outliers (the dots beyond the whiskers), especially in `dti`. These extreme values will need to be handled — either by capping or by applying a log transformation — before training the models.

---
## 10. Correlations

### Why we look at correlations

The correlation matrix tells us two things:
1. Which variables are most correlated with the target (`default`) — these will be the most important features
2. Which variables are highly correlated with each other — this is called **multicollinearity** and can be a problem for logistic regression

In [ ]:
cols_corr = [
    'fico_avg', 'int_rate', 'dti', 'annual_inc', 'revol_util',
    'loan_amnt', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'total_acc', 'grade_num', 'default'
]

corr_matrix = df[cols_corr].corr().round(2)

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Correlation heatmap', fontweight='bold')
plt.tight_layout()
plt.show()


**Finding:** Two important observations from the heatmap:
1. The variables most correlated with `default` are `int_rate` and `grade_num` (positive — higher values mean more default risk) and `fico_avg` (negative — higher FICO means less risk)
2. `int_rate` and `grade_num` have a very high correlation with each other (~0.9). This is multicollinearity — Lending Club sets the interest rate directly based on the grade. This could cause issues for logistic regression. Random Forest and XGBoost handle this naturally.

In [ ]:
# Rank variables by their correlation with the target
corr_with_default = corr_matrix['default'].drop('default').sort_values()

plt.figure(figsize=(8, 6))
sns.barplot(
    x=corr_with_default.values,
    y=corr_with_default.index,
    palette=['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_default.values]
)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation of each variable with the target', fontweight='bold')
plt.xlabel('Pearson Correlation')
plt.ylabel('')
plt.tight_layout()
plt.show()

**Finding:** The ranking confirms the most important predictors. Red bars are positively correlated with default (higher value = more risk): `int_rate`, `grade_num`, `dti`, `delinq_2yrs`. Green bars are negatively correlated (higher value = less risk): `fico_avg`, `annual_inc`, `total_acc`. This ranking gives us a preview of which variables the models will rely on most.

---
## 11. EDA Summary

### Key findings from the full analysis

In [ ]:
print('=' * 60)
print('EDA SUMMARY — Lending Club Credit Risk')
print('=' * 60)
print(f'Loans analyzed           : {len(df):,.0f}')
print(f'Default rate             : {df["default"].mean():.1%}')
print(f'No Default               : {(df["default"]==0).sum():,.0f} ({(df["default"]==0).mean():.1%})')
print(f'Default                  : {(df["default"]==1).sum():,.0f} ({(df["default"]==1).mean():.1%})')
print()
print('Top predictors of default:')
print('  1. int_rate    — interest rate (proxy for LC-assessed risk)')
print('  2. grade       — risk grade assigned by Lending Club')
print('  3. fico_avg    — FICO credit score at origination')
print('  4. revol_util  — revolving credit utilization')
print('  5. dti         — debt-to-income ratio')
print()
print('Key findings:')
print('  - Dataset is imbalanced: 80% no default vs 20% default')
print('  - Grade G has ~6x more default rate than Grade A')
print('  - small_business is the riskiest loan purpose')
print('  - Verified income paradoxically shows higher default (confounder)')
print('  - int_rate and grade_num are highly correlated (~0.9)')
print()
print('Next step: Part 2 — Feature Engineering + Modeling')
print('  - Logistic Regression (baseline)')
print('  - Random Forest')
print('  - XGBoost')
print('  - Evaluation: AUC-ROC, Gini, precision-recall')
print('=' * 60)

In [ ]:
# Save the clean dataframe to use in Part 2 (modeling)
df.to_csv('lending_club_sample.csv', index=False)
print('File saved: lending_club_sample.csv')
print('Shape:', df.shape)

**Final conclusion:** The EDA gave us a solid understanding of the dataset. We know which variables are the best predictors of default, how they are distributed, how we handled missing values, and what transformations we still need before modeling. The main challenge going into Part 2 will be the class imbalance — we will need specific strategies to ensure the models learn to detect defaults rather than just predict the majority class.